#Conditional Variational Autoencoder (C-VAE) Implementation on MNIST Dataset

This notebook implements a CNN-based Variational Autoencoder for the MNIST dataset. We'll follow the concepts presented in the slides to build, train, and evaluate a
C-VAE model.
## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# Set random seed for reproducibility
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 2. Data Loading and Preprocessing

In [ ]:
# MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    # Threshold at 0.5 to create binary images
    lambda x: (x > 0.5).float().clamp(0,1)
])


#train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

# Training dataset
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)

# Test dataset
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Display some example images
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(train_dataset[i][0].squeeze().numpy(), cmap='gray')
    plt.title(f"Label: {train_dataset[i][1]}")
    plt.axis('off')
plt.tight_layout()
plt.show()


## 3. Reconstruction Loss Definition
According to the slides, for binary data like MNIST, we use Binary Cross-Entropy loss which corresponds to a Bernoulli likelihood model:

In [ ]:
def reconstruction_loss(x, x_recon):
    """
    Binary Cross-Entropy loss for Bernoulli likelihood model.

    For binary data (MNIST):
    -log p(x|z) = -∑[x_i * log(θ_i(z)) + (1-x_i) * log(1-θ_i(z))]

    where θ_i(z) is the probability of pixel i being 1, decoded from latent code z.
    This matches the Bernoulli likelihood description from slide 133.
    """
    # BCELoss expects inputs in range [0, 1]
    # Each pixel value in x_recon is interpreted as the probability of that pixel being 1
    BCE = F.binary_cross_entropy(x_recon, x, reduction='sum')
    return BCE


## 4. KL Divergence Loss Definition
From the slides, the KL divergence between our approximate posterior q(z|x) and the prior p(z) acts as a regularization term:

In [ ]:
def kl_divergence(mu, logvar):
    """
    KL divergence between q(z|x) = N(μ(x), σ²(x)) and p(z) = N(0, I).

    As shown in slide 134:
    D_KL(q(z|x) || p(z)) = 1/2 ∑(μ² + σ² - log(σ²) - 1)

    For the standard normal prior p(z) = N(0, I), this simplifies to the formula below.
    """
    # -0.5 * sum(1 + log(σ²) - μ² - σ²)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return KLD


## 5. KL Annealing Implementation
As mentioned in the slides, KL annealing is a technique to gradually increase the weight of the KL term during training, which helps prevent posterior collapse:

In [ ]:
def get_beta(step, warmup_steps, max_beta=1.0):
    """
    Linear KL annealing schedule (as shown in slide 142).

    Args:
        step: Current training step
        warmup_steps: Number of warmup steps
        max_beta: Maximum β value (default: 1.0)

    Returns:
        β value for the current step
    """
    if step > warmup_steps:
        return max_beta
    else:
        return max_beta * (step / warmup_steps)


## 6. CNN-Based VAE Model Implementation
Using the architecture guidelines from the slides (specifically slide 128 for CNN architectures):

In [ ]:



class CVAE(nn.Module):
    def __init__(self, latent_dim=32, num_classes=10, embed_dim=10):
        super(CVAE, self).__init__()
        self.latent_dim = latent_dim
        self.num_classes = num_classes
        self.embed_dim = embed_dim

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 8, 4, 2, 1),  # 28x28 -> 14x14
            nn.ReLU(),
            nn.Conv2d(8, 16, 4, 2, 1),  # 14x14 -> 7x7
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, 1, 1),  # 7x7 -> 7x7
            nn.ReLU(),
            nn.Flatten()
        )

        self.enc_out_dim = 32 * 7 * 7
        self.label_embed = nn.Embedding(num_classes, embed_dim)

        # Mean and logvar layers (concat image features + label embedding)
        self.fc_mu = nn.Linear(self.enc_out_dim + embed_dim, latent_dim)
        self.fc_logvar = nn.Linear(self.enc_out_dim + embed_dim, latent_dim)

        # Decoder
        self.decoder_input = nn.Linear(latent_dim + embed_dim, 7*7*16)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(16, 16, 4, 2, 1),  # 7x7 -> 14x14
            nn.ReLU(),
            nn.ConvTranspose2d(16, 8, 4, 2, 1),  # 14x14 -> 28x28
            nn.ReLU(),
            nn.ConvTranspose2d(8, 1, 3, 1, 1),    #  28x28 -> 28x28
            nn.Sigmoid()
        )

    def encode(self, x, c):
        # Image encoding
        h = self.encoder(x)
        # Label embedding
        c_emb = self.label_embed(c)
        # Concatenate
        h = torch.cat([h, c_emb], dim=1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        logvar = torch.clamp(logvar, min=-10, max=10)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5*logvar)
        eps = torch.randn_like(std)
        return mu + eps*std

    def decode(self, z, c):
        # Label embedding
        c_emb = self.label_embed(c)
        # Concatenate
        z = torch.cat([z, c_emb], dim=1)
        h = self.decoder_input(z)
        h = h.view(-1, 16, 7, 7)
        return self.decoder(h)

    def forward(self, x, c):
        mu, logvar = self.encode(x, c)
        z = self.reparameterize(mu, logvar)
        return self.decode(z, c), mu, logvar

# Initialize model
latent_dim = 32
embed_dim = 10
num_classes = 10

model = CVAE(latent_dim, num_classes, embed_dim).to(device)
print(model)


## 7. Training Loop with KL Annealing

In [ ]:
def train(model, train_loader, optimizer, epochs=30, warmup_steps=10000, max_beta=1.0):
    model.train()
    train_losses = []
    kl_losses = []
    recon_losses = []
    step = 0

    for epoch in range(epochs):
        epoch_loss = 0
        epoch_recon_loss = 0
        epoch_kl_loss = 0

        progress_bar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{epochs}")

        for i, (x, c) in progress_bar:

            x = x.to(device)
            c = c.to(device)
            optimizer.zero_grad()

            # Forward pass
            x_recon, mu, logvar = model(x, c)

            # Calculate losses
            recon_loss = reconstruction_loss(x, x_recon)
            kl_loss = kl_divergence(mu, logvar)

            # Apply KL annealing
            beta = get_beta(step, warmup_steps, max_beta)

            # Total loss (following the ELBO objective from slide 133)
            loss = recon_loss + beta * kl_loss

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            # Update metrics
            epoch_loss += loss.item()
            epoch_recon_loss += recon_loss.item()
            epoch_kl_loss += kl_loss.item()

            # Update progress bar
            progress_bar.set_postfix({
                'loss': loss.item() / len(x),
                'recon_loss': recon_loss.item() / len(x),
                'kl_loss': kl_loss.item() / len(x),
                'beta': beta
            })

            step += 1

        # Compute average losses
        avg_loss = epoch_loss / len(train_loader.dataset)
        avg_recon_loss = epoch_recon_loss / len(train_loader.dataset)
        avg_kl_loss = epoch_kl_loss / len(train_loader.dataset)

        train_losses.append(avg_loss)
        recon_losses.append(avg_recon_loss)
        kl_losses.append(avg_kl_loss)

        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Recon Loss: {avg_recon_loss:.4f}, KL Loss: {avg_kl_loss:.4f}, Beta: {beta:.4f}")

    return train_losses, recon_losses, kl_losses

# Initialize optimizer (Adam as recommended in the slides)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Train model
print("Starting training...")
train_losses, recon_losses, kl_losses = train(model, train_loader, optimizer, epochs=16, warmup_steps=2000)
print("Training complete!")


## 8. Plot Training Metrics

In [ ]:
# Plot training losses
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(train_losses)
plt.title('Total Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 3, 2)
plt.plot(recon_losses)
plt.title('Reconstruction Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 3, 3)
plt.plot(kl_losses)
plt.title('KL Divergence Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.tight_layout()
plt.show()

## 9. Evaluate and Visualize Reconstructions

In [ ]:
def visualize_reconstructions(model, test_loader, n=10):
    model.eval()

    # Get batch of test images
    dataiter = iter(test_loader)
    images, labels = next(dataiter)
    images = images[:n].to(device)
    labels = labels[:n].to(device)

    # Reconstruct images
    with torch.no_grad():
        reconstructions, _, _ = model(images, labels)

    # Plot original vs reconstructed images
    plt.figure(figsize=(20, 4))
    for i in range(n):
        # Original images
        ax = plt.subplot(2, n, i + 1)
        plt.imshow(images[i].cpu().squeeze().numpy(), cmap='gray')
        plt.title(f"Original: {labels[i]}")
        plt.axis('off')

        # Reconstructed images
        ax = plt.subplot(2, n, n + i + 1)
        plt.imshow(reconstructions[i].cpu().squeeze().numpy(), cmap='gray')
        plt.title(f"Reconstructed")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Visualize reconstructions
visualize_reconstructions(model, test_loader)


## 10. Generate Samples from the Latent Space
As described in slides, we can sample from the prior distribution and decode to generate new samples:

In [ ]:
# Generate conditioned samples
def generate_conditioned_samples(model, label, num_samples=10):
    model.eval()
    with torch.no_grad():
        c = torch.tensor([label]*num_samples).to(device)
        z = torch.randn(num_samples, latent_dim).to(device)
        samples = model.decode(z, c)

        plt.figure(figsize=(10, 2))
        for i in range(num_samples):
            plt.subplot(1, num_samples, i+1)
            plt.imshow(samples[i].cpu().squeeze(), cmap='gray')
            plt.axis('off')
        plt.show()
# Generate samples
# Generate samples for digit 3
generate_conditioned_samples(model, label=0, num_samples=10)


### TODO-1: Latent Traversals

For a trained C-VAE, fix all latent variables except one and vary that dimension.
- Visualize the generated images as you sweep through the latent dimension.
- Discuss: Do certain dimensions correspond to interpretable features


In [ ]:
# TODO: Implement latent traversal visualization
# For each dimension, vary z_i in [-3, 3] while fixing others


## TODO-2: Generate Your Roll Number

Use the trained C-VAE to generate a sequence of digits that corresponds to your roll number.
For example, if your roll number is `660700`, you should generate the digits `6`, `6`, `0`, `7`, `0`, `0` in sequence using the C-VAE conditioned on those labels.

- Condition the C-VAE on the labels corresponding to your roll number digits.
- **Generate your roll number 5 times** by sampling different latent vectors $z$ for each digit.
- Plot the generated images as a grid (5 rows, one for each generation) to see how the style changes each time.
- Discuss the variations you observe in the generated digits (e.g., thickness, slant).

In [ ]:
# TODO: Write your code here to generate your roll number
# Hint: You can use the `generate_conditioned_samples` function defined above.

## TODO-3: VAE on CIFAR-100


Replace the MNIST dataset with CIFAR-100.
- Adjust the network architecture if needed.
- Train and evaluate your VAE on the new dataset.
- Compare reconstruction and generation quality with your MNIST results.
- Goal is to study how C-VAE works for 100 classes

**Hint: BCE may not be the best option for CIFAR-100**

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import random

# Define transformation: ToTensor() converts images to [0,1] float tensors
transform = transforms.Compose([
    transforms.ToTensor()
])

# Load Fashion-MNIST training and test datasets
train_dataset = datasets.CIFAR100(
    root='data',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR100(
    root='data',
    train=False,
    download=True,
    transform=transform
)

# Create DataLoaders
batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

# Display some example images
plt.figure(figsize=(10, 4))
for idx, i in enumerate(random.sample(range(128),10)):
    plt.subplot(2, 5, idx + 1)
    plt.imshow(train_dataset[i][0].permute(1, 2, 0).numpy())
    plt.title(f"Label: {train_dataset[i][1]}")
    plt.axis('off')
plt.tight_layout()
plt.show()

## 📝 Student Information

Please fill in your details below before submitting.

- **Student Name:** *Type your name here*
- **Roll Number:** *Type your roll number here*
- **Date of Completion:** *Type date here*

## 📤 Submission Instructions

Please submit your completed `.ipynb` notebook file to the Google Form linked below:

🔗 [Submit Here](https://docs.google.com/forms/d/e/1FAIpQLSeEHrefPva1f6_TGCoW-yjmp_NnAb0Be2a12mYpoKS-NtLc4A/viewform?usp=dialog)

⚠️ **Deadline:** March 29, 2026, 11:59:00 PM GMT+5:30